# QualityMind: Train YOLO26 on NEU-DET for Industrial Surface Defect Detection

This notebook trains Ultralytics **YOLO26** (AGPL-3.0 License) on the public **NEU-DET** (Northeastern University Surface Defect Database) on a **free Colab or Kaggle GPU (T4)** and exports an ultra-lightweight **ONNX** model.

### Target Hardware:
- Training: Free Colab T4 GPU (Runtime -> Change runtime type -> T4 GPU)
- Production: Render Free Tier CPU (< 512 MB RAM, < 100 ms per image via ONNX Runtime)

### Defect Classes:
1. `crazing`
2. `inclusion`
3. `patches`
4. `pitted_surface`
5. `rolled-in_scale`
6. `scratches`

In [ ]:
# Step 1: Install Ultralytics and dependencies
!pip install -q ultralytics onnx onnxruntime onnxslim roboflow

In [ ]:
# Step 2: Download NEU-DET Surface Defect Dataset
!git clone https://github.com/csu-xuchao/NEU-DET.git || echo 'Already cloned'
import os
print('Dataset files:', len(os.listdir('NEU-DET')) if os.path.exists('NEU-DET') else 'Ready')

In [ ]:
# Step 3: Create YAML Dataset Configuration
import yaml

data_yaml = {
    'names': ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches'],
    'nc': 6,
    'train': '/content/dataset/images/train',
    'val': '/content/dataset/images/val'
}

with open('neu_det.yaml', 'w') as f:
    yaml.dump(data_yaml, f)
print('Configuration written to neu_det.yaml')

In [ ]:
# Step 4: Train YOLO26 Nano Model with Transfer Learning
from ultralytics import YOLO

model = YOLO('yolo26n.pt')  # Ultralytics YOLO26
results = model.train(
    data='neu_det.yaml',
    epochs=30,
    imgsz=512,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    device=0
)

In [ ]:
# Step 5: Export to Lightweight ONNX Model for Render (<512 MB RAM Free Tier)
onnx_path = model.export(format='onnx', imgsz=512, simplify=True, opset=17)
print(f'ONNX Model exported: {onnx_path}')

# Download the file to local computer for QualityMind deployment
from google.colab import files
files.download(onnx_path)